# 국내 지역 지수(Domestic Regional Index) — 최종 코퍼스 10,020건 산출 노트북

국내 지역 지수는 근거문장 안의 17개 시/도 지역명 언급을 세는 텍스트마이닝 보조지표다(Coverage Index의 해외 market_coverage와 별개).
이 노트북은 키워드 규칙을 최종 코퍼스 10,020건에 적용해 `domestic_regional_index_v7.json` / `.csv`를 산출하고,
보고서 표 15(지역 언급 상위 20개 아티스트)가 **동결 스냅샷 v7-40(7,350건)** 시점 값임을 동결 코퍼스 근사로 확인한다.

```
RegionMentionCount(fandom, region) = 해당 팬덤 loyalty+spillover 근거문장 중 지역 키워드를 포함하는 불릿 수
RegionDiversity(fandom)            = n_regions_hit(검출된 서로 다른 지역 수) ÷ 17
```

In [1]:
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 140)


def find_repo_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "data" / "v7_final" / "fandoms_v3_100.json").exists():
            return p
    raise FileNotFoundError("저장소 루트(data/v7_final/fandoms_v3_100.json)를 찾지 못함 — 저장소 안에서 실행하세요")


REPO = find_repo_root()
DATA_DIR = REPO / "data" / "v7_final"                       # 최종 산출물(10,020건 라이브 + 동결 스냅샷 7,350건)
ROUNDS_DIR = REPO / "data" / "v7_rounds"                    # 병합 로그 r1~r72


def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def flatten_bullets(fandoms):
    rows = []
    for rec in fandoms:
        for kind in ("loyalty", "spillover"):
            for item in rec.get(kind, []):
                rows.append({"fandom": rec["fandom"], "category": rec.get("category"),
                             "bullet_type": kind, "text": item.get("t", "") or "", "url": item.get("u", "") or ""})
    return pd.DataFrame(rows)

REGIONS = ["서울", "부산", "대구", "인천", "광주", "대전", "울산", "세종", "경기", "강원",
           "충북", "충남", "전북", "전남", "경북", "경남", "제주"]
# 17개 시/도 키워드 사전(METHODOLOGY 국내 지역 지수 규칙). 세종은 "세종문화회관" 오매칭 방지로 "세종시/세종특별자치시"만,
# 경기는 일상어 "경기"와의 충돌로 "경기도"+주요 시 명칭만 인정. 고양·여주·완주·동해·예산처럼 일상어와 충돌하는 지명은 제외.
REGION_KEYWORDS = {
    "서울": ["서울"], "부산": ["부산"], "대구": ["대구"], "인천": ["인천"], "광주": ["광주"], "대전": ["대전"], "울산": ["울산"],
    "세종": ["세종시", "세종특별자치시"],
    "경기": ["경기도", "수원", "성남", "부천", "안산", "화성", "의정부", "남양주"],
    "강원": ["강원", "춘천", "강릉", "원주", "속초"],
    "충북": ["충북", "청주"],
    "충남": ["충남", "천안", "아산", "보령"],
    "전북": ["전북", "전주"],
    "전남": ["전남", "목포", "순천", "진도", "전라남도"],
    "경북": ["경북", "포항", "경주", "김천", "구미", "울릉"],
    "경남": ["경남", "창원", "진주", "김해", "거제", "남해"],
    "제주": ["제주"],
}
NOTE = ("TEXT-MINING INDEX: 이미 수집된 근거문장 내 국내 지역명(17개 시/도 및 대표 도시) 언급 불릿 수 기반. "
        "Coverage Index의 market_coverage(해외 시장)와는 별개 지표이며, 지역 연고(고향)와 지역 활동(투어·행사)을 구분하지 않고 합산한 1차 신호다. "
        "최종 코퍼스 10,020건 기준 산출.")


def region_index(fandoms):
    out = {}
    for rec in fandoms:
        texts = [b.get("t", "") or "" for k in ("loyalty", "spillover") for b in rec.get(k, [])]
        counts = {r: sum(1 for t in texts if any(k in t for k in REGION_KEYWORDS[r])) for r in REGIONS}
        total = sum(counts.values())
        hit = sum(1 for v in counts.values() if v > 0)
        primary = max(counts, key=counts.get) if total else None
        out[rec["fandom"]] = {
            "total_group_bullets": len(texts), "region_mention_counts": counts, "total_region_mentions": total,
            "n_regions_hit": hit, "region_diversity": round(hit / 17, 3),
            "primary_region": primary, "primary_region_share": round(counts[primary] / total, 3) if total else 0.0,
            "note": NOTE,
        }
    return out

fandoms = load_json(DATA_DIR / "fandoms_v3_100.json")
print("최종 코퍼스:", sum(len(f.get("loyalty", [])) + len(f.get("spillover", [])) for f in fandoms), "건 |", len(fandoms), "팬덤")
print("지역 키워드 수:", sum(len(v) for v in REGION_KEYWORDS.values()))

최종 코퍼스: 10020 건 | 100 팬덤
지역 키워드 수: 48


## 1. 최종 코퍼스 10,020건 산출 — JSON/CSV

In [2]:
final = region_index(fandoms)
OUT_DIR = Path.cwd() if (Path.cwd() / "domestic_regional_index_v7.ipynb").exists() else REPO / "v7_final_10020" / "analysis" / "domestic_regional_index"
with open(OUT_DIR / "domestic_regional_index_v7.json", "w", encoding="utf-8") as f:
    json.dump(final, f, ensure_ascii=False, indent=2)
rows = []
for name, d in final.items():
    row = {"팬덤": name, "근거문장수": d["total_group_bullets"], "총지역언급": d["total_region_mentions"], "검출지역수": d["n_regions_hit"],
           "지역다양성": d["region_diversity"], "대표지역": d["primary_region"] or "", "대표지역비중": d["primary_region_share"]}
    row.update(d["region_mention_counts"]); rows.append(row)
dr_df = pd.DataFrame(rows).sort_values(["총지역언급", "팬덤"], ascending=[False, True]).reset_index(drop=True)
dr_df.to_csv(OUT_DIR / "domestic_regional_index_v7.csv", index=False, encoding="utf-8-sig")
dr_df.index += 1
n_zero = int((dr_df["총지역언급"] == 0).sum())
print("저장: v7_final_10020/analysis/domestic_regional_index/domestic_regional_index_v7.json / .csv")
print(f"팬덤 커버리지: {100 - n_zero}/100 (0건 팬덤 {n_zero}개: {list(dr_df[dr_df['총지역언급'] == 0]['팬덤'])})")
print(f"검출 지역 수 합계(n_regions_hit 합): {int(dr_df['검출지역수'].sum())} / 1,700 | 총 지역언급 {int(dr_df['총지역언급'].sum())}건")
dr_df.head(18)

저장: v7_final_10020/analysis/domestic_regional_index/domestic_regional_index_v7.json / .csv
팬덤 커버리지: 99/100 (0건 팬덤 1개: ['투어스(TWS)'])
검출 지역 수 합계(n_regions_hit 합): 469 / 1,700 | 총 지역언급 1226건


,팬덤,근거문장수,총지역언급,검출지역수,지역다양성,대표지역,대표지역비중,서울,부산,대구,...,세종,경기,강원,충북,충남,전북,전남,경북,경남,제주
1,싸이,122,43,10,0.588,강원,0.163,6,6,5,...,0,5,7,0,0,3,0,1,0,0
2,이승철,78,38,9,0.529,경기,0.158,5,4,4,...,0,6,4,5,0,0,0,0,0,0
3,박서진,99,31,11,0.647,서울,0.258,8,3,4,...,0,1,0,1,1,0,0,4,3,0
4,송가인,92,31,6,0.353,전남,0.742,1,0,1,...,0,0,0,0,0,3,23,2,0,0
5,나훈아,70,30,9,0.529,서울,0.333,10,6,5,...,0,0,0,1,0,0,0,1,1,0
6,god,99,28,4,0.235,서울,0.536,15,8,3,...,0,0,0,0,0,0,0,0,0,0
7,임영웅,131,28,12,0.706,서울,0.214,6,3,2,...,0,3,1,0,1,0,0,1,1,2
8,빅마마,73,26,7,0.412,서울,0.269,7,5,4,...,0,0,0,0,0,4,0,1,0,0
9,거미,87,25,12,0.706,서울,0.200,5,1,2,...,0,4,0,0,1,1,2,0,1,1
10,김연자,73,25,8,0.471,광주,0.280,6,4,1,...,0,2,0,0,0,1,3,0,0,0


## 2. 보고서 표 15(지역 언급 상위 20개)와의 대조 — 표 15는 동결 스냅샷(7,350건) 값

최종 10,020건 값은 표 15보다 큰 팬덤이 많다(god 16→28, 싸이 35→43 등). 동결 스냅샷 코퍼스 파일은 저장소에 없지만, 병합이 팬덤별로 뒤에 덧붙는
구조이므로 **각 팬덤의 loyalty/spillover 앞쪽 n건(동결 점수 JSON의 n_loyalty_bullets/n_spillover_bullets)**을 취하면 동결 코퍼스를 근사할 수 있다.

In [3]:
REPORT_TABLE15 = {  # (총언급, 검출수, 대표지역, 비중, 다양성) — 분석보고서 표 15
    "싸이": (35, 10, "강원", 0.20, 0.59), "이승철": (32, 9, "서울", 0.13, 0.53), "임영웅": (28, 12, "서울", 0.21, 0.71), "송가인": (27, 6, "전남", 0.74, 0.35),
    "박서진": (22, 11, "서울", 0.23, 0.65), "악동뮤지션": (21, 9, "서울", 0.33, 0.53), "김연자": (21, 8, "광주", 0.33, 0.47), "나훈아": (20, 8, "서울", 0.35, 0.47),
    "조용필": (20, 6, "서울", 0.30, 0.35), "리센느(RESCENE)": (19, 5, "경남", 0.58, 0.29), "이영지": (18, 10, "서울", 0.39, 0.59), "다이나믹듀오": (18, 6, "서울", 0.28, 0.35),
    "영탁": (18, 7, "서울", 0.33, 0.41), "지드래곤 (G-Dragon)": (17, 4, "서울", 0.47, 0.23), "BTS": (17, 5, "서울", 0.47, 0.29), "god": (16, 4, "서울", 0.50, 0.23),
    "잭스키스": (15, 5, "부산", 0.53, 0.29), "로이킴": (15, 6, "서울", 0.53, 0.35), "김호중": (14, 6, "경북", 0.43, 0.35), "엄정화": (14, 3, "부산", 0.36, 0.18),
}
frozen_scores = load_json(DATA_DIR / "fandom_scores_v6.json")
fz = {d["fandom"]: d for d in frozen_scores}
frozen_approx_fandoms = []
for rec in fandoms:
    d = fz.get(rec["fandom"])
    if d is None:
        continue
    frozen_approx_fandoms.append({"fandom": rec["fandom"], "loyalty": rec["loyalty"][:d["n_loyalty_bullets"]], "spillover": rec["spillover"][:d["n_spillover_bullets"]]})
frozen_approx = region_index(frozen_approx_fandoms)
print(f"동결 근사 코퍼스: {sum(v['total_group_bullets'] for v in frozen_approx.values())}건 / {len(frozen_approx)}팬덤 (동결 점수 JSON activity 합 {sum(d['activity'] for d in frozen_scores)})")
print(f"동결 근사: 검출 지역 수 합 {sum(v['n_regions_hit'] for v in frozen_approx.values())}, 커버리지 {sum(1 for v in frozen_approx.values() if v['n_regions_hit'] > 0)}/{len(frozen_approx)}, "
      f"서울 비중 {sum(v['region_mention_counts']['서울'] for v in frozen_approx.values()) / sum(v['total_region_mentions'] for v in frozen_approx.values()):.1%}")

rows = []; n_match_frozen = n_match_final = 0
for name, (tot, hit, reg, share, div) in REPORT_TABLE15.items():
    fa, fi = frozen_approx[name], final[name]
    m_f = fa["total_region_mentions"] == tot and fa["n_regions_hit"] == hit and fa["primary_region"] == reg and abs(fa["primary_region_share"] - share) < 0.011
    m_l = fi["total_region_mentions"] == tot and fi["n_regions_hit"] == hit and fi["primary_region"] == reg and abs(fi["primary_region_share"] - share) < 0.011
    n_match_frozen += m_f; n_match_final += m_l
    rows.append({"팬덤": name, "표15 총언급/검출/대표(비중)": f"{tot}/{hit}/{reg}({share:.0%})",
                 "동결근사": f"{fa['total_region_mentions']}/{fa['n_regions_hit']}/{fa['primary_region']}({fa['primary_region_share']:.0%})", "동결근사 일치": m_f,
                 "최종 10,020": f"{fi['total_region_mentions']}/{fi['n_regions_hit']}/{fi['primary_region']}({fi['primary_region_share']:.0%})", "최종 일치": m_l})
print(f"표 15 20행 중 정확 일치 — 동결 근사: {n_match_frozen}/20, 최종 10,020건: {n_match_final}/20")
print("-> 표 15는 동결 스냅샷 값이고, 나머지 3행(이영지·영탁·로이킴)은 근사 코퍼스가 앞쪽 n건 가정과 1~2건 어긋난 결과다(라운드 스왑·중복 제거 흔적).")
for name in ["BTS", "임영웅", "리센느(RESCENE)"]:
    d = final[name]
    print(f"최종 {name}: 총언급 {d['total_region_mentions']}, 검출 {d['n_regions_hit']}, 대표 {d['primary_region']}({d['primary_region_share']:.0%}), 다양성 {d['region_diversity']}")
top_div = dr_df.sort_values(["지역다양성", "총지역언급"], ascending=[False, False]).iloc[0]
print(f"최종 지역다양성 전체 1위: {top_div['팬덤']} ({top_div['지역다양성']})  (보고서: 임영웅 0.71 전체 1위)")
pd.DataFrame(rows)

동결 근사 코퍼스: 7231건 / 97팬덤 (동결 점수 JSON activity 합 7350)
동결 근사: 검출 지역 수 합 410, 커버리지 97/97, 서울 비중 35.1%
표 15 20행 중 정확 일치 — 동결 근사: 17/20, 최종 10,020건: 5/20
-> 표 15는 동결 스냅샷 값이고, 나머지 3행(이영지·영탁·로이킴)은 근사 코퍼스가 앞쪽 n건 가정과 1~2건 어긋난 결과다(라운드 스왑·중복 제거 흔적).
최종 BTS: 총언급 17, 검출 5, 대표 서울(47%), 다양성 0.294
최종 임영웅: 총언급 28, 검출 12, 대표 서울(21%), 다양성 0.706
최종 리센느(RESCENE): 총언급 22, 검출 5, 대표 경남(64%), 다양성 0.294
최종 지역다양성 전체 1위: 임영웅 (0.706)  (보고서: 임영웅 0.71 전체 1위)


,팬덤,표15 총언급/검출/대표(비중),동결근사,동결근사 일치,"최종 10,020",최종 일치
0,싸이,35/10/강원(20%),35/10/강원(20%),True,43/10/강원(16%),False
1,이승철,32/9/서울(13%),32/9/서울(12%),True,38/9/경기(16%),False
2,임영웅,28/12/서울(21%),28/12/서울(21%),True,28/12/서울(21%),True
3,송가인,27/6/전남(74%),27/6/전남(74%),True,31/6/전남(74%),False
4,박서진,22/11/서울(23%),22/11/서울(23%),True,31/11/서울(26%),False
5,악동뮤지션,21/9/서울(33%),21/9/서울(33%),True,21/9/서울(33%),True
6,김연자,21/8/광주(33%),21/8/광주(33%),True,25/8/광주(28%),False
7,나훈아,20/8/서울(35%),20/8/서울(35%),True,30/9/서울(33%),False
8,조용필,20/6/서울(30%),20/6/서울(30%),True,20/6/서울(30%),True
9,리센느(RESCENE),19/5/경남(58%),19/5/경남(58%),True,22/5/경남(64%),False


## 3. 지역별 전국 분포

In [4]:
region_totals = {r: int(dr_df[r].sum()) for r in REGIONS}
grand = sum(region_totals.values())
dist = pd.DataFrame([{"지역": r, "언급불릿수": c, "비중": round(c / grand, 4), "언급 팬덤 수": int((dr_df[r] > 0).sum())}
                     for r, c in sorted(region_totals.items(), key=lambda x: -x[1])])
print(f"전체 지역 언급 총계: {grand}건")
dist

전체 지역 언급 총계: 1226건


,지역,언급불릿수,비중,언급 팬덤 수
0,서울,484,0.3948,95
1,부산,138,0.1126,48
2,대구,97,0.0791,50
3,인천,83,0.0677,47
4,경기,71,0.0579,37
5,경북,47,0.0383,24
6,광주,46,0.0375,24
7,경남,46,0.0375,22
8,대전,36,0.0294,22
9,강원,36,0.0294,24


## 4. 한계

1. 보고서 표 15·KEY_FINDINGS의 국내 지역 값은 **동결 스냅샷(7,350건)** 기준이며, 이 폴더의 `domestic_regional_index_v7.json`은 **최종 10,020건** 기준이라 상위 팬덤 수치가 더 크다.
2. 부분 문자열 매칭이라 "광주"(경기 광주시 vs 광주광역시), "고성"(강원/경남) 같은 동음 지명은 구분하지 않는다.
3. 언급 0건은 "이 코퍼스에서 검출되지 않았다"는 뜻이지 지역 연고가 없다는 뜻이 아니다.